# Colorado groundwater levels (USGS)

Mirrors the `snow_pack` notebook, but for USGS groundwater level data in Colorado.

Uses the **USGS Water Data OGC API** (`api.waterdata.usgs.gov`). The legacy `waterservices.usgs.gov` endpoints (`/iv`, `/dv`) are scheduled for decommission in early 2027 and `gwlevels` is already gone, so this avoids them.

- Parameter `72019` = depth to water level, **feet below land surface**. Bigger number = deeper water = falling water table, so the plots invert the y-axis.
- Collections used: `monitoring-locations`, `time-series-metadata`, `daily`, `continuous`, `field-measurements`, `latest-daily`, `latest-continuous`.
- Optional: set a free API key in the `USGS_API_KEY` environment variable for higher rate limits (https://api.waterdata.usgs.gov/signup).

In [ ]:
import os
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta
from pathlib import Path

In [ ]:
BASE_URL = "https://api.waterdata.usgs.gov/ogcapi/v0"
API_KEY = os.environ.get("USGS_API_KEY")  # optional
GW_PARAM = "72019"  # depth to water level, ft below land surface

def chunks(seq, n):
    seq = list(seq)
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def ogc_get(collection, max_pages=100, **params):
    """GET /collections/{collection}/items, follow pagination, return a DataFrame.
    Feature properties become columns; point geometry becomes longitude/latitude;
    the feature id is kept as `feature_id`."""
    clean = {}
    for k, v in params.items():
        if v is None:
            continue
        if isinstance(v, bool):
            v = str(v).lower()
        elif isinstance(v, (list, tuple)):
            v = ",".join(map(str, v))
        clean[k] = v
    clean.setdefault("f", "json")
    clean.setdefault("limit", 10000)

    headers = {"X-Api-Key": API_KEY} if API_KEY else {}
    url, query = f"{BASE_URL}/collections/{collection}/items", clean
    rows, pages = [], 0
    while url and pages < max_pages:
        resp = requests.get(url, params=query, headers=headers, timeout=60)
        resp.raise_for_status()
        js = resp.json()
        for feat in js.get("features", []):
            row = dict(feat.get("properties") or {})
            row["feature_id"] = feat.get("id")
            geom = feat.get("geometry") or {}
            if geom.get("type") == "Point":
                row["longitude"], row["latitude"] = geom["coordinates"][:2]
            rows.append(row)
        # the "next" link already carries the full query string
        url = next((l["href"] for l in js.get("links", []) if l.get("rel") == "next"), None)
        query = None
        pages += 1
    return pd.DataFrame(rows)

print("Ready. Base URL:", BASE_URL, "| API key set:", bool(API_KEY))

## 1. Find Colorado groundwater sites

In [ ]:
def find_gw_sites(state_name="Colorado", site_type_code="GW", **extra):
    """All monitoring locations of a given site type (GW = groundwater well) in a state."""
    df = ogc_get("monitoring-locations", state_name=state_name,
                 site_type_code=site_type_code, **extra)
    if "monitoring_location_id" not in df.columns:
        df["monitoring_location_id"] = df["feature_id"]
    return df

co_sites = find_gw_sites()
print(f"{len(co_sites)} Colorado groundwater monitoring locations")
show = [c for c in ["monitoring_location_id", "monitoring_location_name", "altitude",
                    "latitude", "longitude"] if c in co_sites.columns]
co_sites[show].head(10)

## 2. Which sites actually have a water-level time series?

`time-series-metadata` tells us the period of record (`begin`/`end`) per series, so we can find wells that are still reporting.

In [ ]:
def find_gw_series(site_ids, parameter_code=GW_PARAM, per_request=50):
    frames = []
    for chunk in chunks(site_ids, per_request):
        frames.append(ogc_get("time-series-metadata",
                              monitoring_location_id=chunk,
                              parameter_code=parameter_code,
                              skipGeometry=True))
    frames = [f for f in frames if not f.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

gw_series = find_gw_series(co_sites["monitoring_location_id"])
for c in ("begin", "end"):
    gw_series[c] = pd.to_datetime(gw_series[c], utc=True, errors="coerce")

print(f"{len(gw_series)} series across {gw_series['monitoring_location_id'].nunique()} sites")
if "computation_identifier" in gw_series.columns:
    print(gw_series["computation_identifier"].value_counts())

gw_summary = (gw_series.groupby("monitoring_location_id")
              .agg(begin=("begin", "min"), end=("end", "max"), n_series=("begin", "size"))
              .reset_index()
              .merge(co_sites[show], on="monitoring_location_id", how="left"))
gw_summary["record_years"] = (gw_summary["end"] - gw_summary["begin"]).dt.days / 365.25
gw_summary.sort_values("end", ascending=False).head(10)

In [ ]:
# Pick the longest record among wells that reported in the last 90 days
cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=90)
active = gw_summary[gw_summary["end"] >= cutoff].sort_values("record_years", ascending=False)
print(f"{len(active)} sites reported in the last 90 days")

example_site = active.iloc[0]["monitoring_location_id"] if len(active) else gw_summary.iloc[0]["monitoring_location_id"]
print("Using site:", example_site, "-", gw_summary.set_index("monitoring_location_id").loc[example_site].get("monitoring_location_name"))

## 3. Pull data for a site

`collection` can be `daily` (one value/day), `continuous` (sub-daily sensor readings, last ~3 years online), or `field-measurements` (manual tape-downs, usually sparse but with a long history).

In [ ]:
def get_gw_data(site_ids, collection="daily", parameter_code=GW_PARAM,
                begin=None, end=None, per_request=25):
    """
    site_ids   : str or list[str], e.g. "USGS-390000106000001"
    collection : "daily" | "continuous" | "field-measurements"
    begin/end  : "YYYY-MM-DD" (end=None means open-ended)
    """
    if isinstance(site_ids, str):
        site_ids = [site_ids]
    if begin is None:
        begin = (date.today() - timedelta(days=730)).isoformat()
    b = begin + "T00:00:00Z" if (collection == "continuous" and len(begin) == 10) else begin
    e = end if end is None or collection != "continuous" or len(end) != 10 else end + "T23:59:59Z"
    interval = f"{b}/{e or '..'}"

    frames = []
    for chunk in chunks(site_ids, per_request):
        frames.append(ogc_get(collection, monitoring_location_id=chunk,
                              parameter_code=parameter_code, time=interval,
                              skipGeometry=True))
    frames = [f for f in frames if not f.empty]
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.sort_values(["monitoring_location_id", "time"]).reset_index(drop=True)

daily_df = get_gw_data(example_site, "daily", begin="2023-10-01")
print(f"{len(daily_df)} daily rows")
daily_df[["monitoring_location_id", "time", "value", "unit_of_measure",
          "approval_status"]].head() if len(daily_df) else daily_df

In [ ]:
# Depth to water: invert y-axis so a falling water table plots downward
if daily_df.empty:
    print(f"No daily data returned for {example_site} in this date range.")
else:
    if "statistic_id" in daily_df.columns and daily_df["statistic_id"].nunique() > 1:
        print("Multiple statistics present, using mean (00003):", daily_df["statistic_id"].unique())
        daily_df = daily_df[daily_df["statistic_id"] == "00003"]

    field_df = get_gw_data(example_site, "field-measurements", begin=daily_df["time"].min().date().isoformat())

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(daily_df["time"], daily_df["value"], color="tab:blue", label="Daily")
    if not field_df.empty:
        ax.scatter(field_df["time"], field_df["value"], color="tab:red", s=18, zorder=3,
                   label="Field measurement")
    ax.invert_yaxis()
    ax.set_ylabel("Depth to water (ft below land surface)")
    ax.set_title(f"Groundwater level — {example_site}")
    ax.legend()
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## 4. Compare several wells

In [ ]:
sample_sites = active["monitoring_location_id"].head(5).tolist() or [example_site]
multi_df = get_gw_data(sample_sites, "daily", begin="2023-10-01")

if multi_df.empty:
    print("No data returned for these sites/date range.")
else:
    if "statistic_id" in multi_df.columns:
        multi_df = multi_df[multi_df["statistic_id"] == "00003"]
    fig, ax = plt.subplots(figsize=(10, 5))
    for site, g in multi_df.groupby("monitoring_location_id"):
        # plot change relative to each well's first value so differently-deep wells are comparable
        ax.plot(g["time"], g["value"] - g["value"].iloc[0], label=site)
    ax.invert_yaxis()
    ax.set_ylabel("Change in depth to water (ft, + = deeper)")
    ax.set_title("Groundwater level change across wells")
    ax.legend(fontsize=8)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

## 5. Dashboard-ready snapshot

Builds one row per well: latest reading, how recent it is, and the change over the last 30 days. Writes `data/co_gw_latest.csv` and `.json`, which is what a GitHub Pages dashboard can read. Run this cell (or export it as a script) on a schedule to keep the dashboard fresh.

In [ ]:
def latest_values(site_ids, per_request=25):
    """Most recent reading per site; prefers continuous sensor data, falls back to daily."""
    out = []
    for coll in ("latest-continuous", "latest-daily"):
        frames = [ogc_get(coll, monitoring_location_id=c, parameter_code=GW_PARAM, skipGeometry=True)
                  for c in chunks(site_ids, per_request)]
        frames = [f for f in frames if not f.empty]
        if frames:
            df = pd.concat(frames, ignore_index=True)
            df["source"] = coll
            out.append(df)
    if not out:
        return pd.DataFrame()
    df = pd.concat(out, ignore_index=True)
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    # keep the newest reading per site (continuous wins ties because it is listed first)
    df = df.sort_values(["monitoring_location_id", "time"], ascending=[True, False])
    return df.drop_duplicates("monitoring_location_id")[
        ["monitoring_location_id", "time", "value", "source"]
    ].rename(columns={"time": "latest_time", "value": "latest_depth_ft"})

def change_over_window(site_ids, days=30):
    """Depth change vs. `days` ago from daily values (+ = water table deeper / falling)."""
    begin = (date.today() - timedelta(days=days + 5)).isoformat()
    df = get_gw_data(site_ids, "daily", begin=begin)
    if df.empty:
        return pd.DataFrame(columns=["monitoring_location_id", f"change_{days}d_ft"])
    if "statistic_id" in df.columns:
        df = df[df["statistic_id"] == "00003"]
    rows = []
    for site, g in df.groupby("monitoring_location_id"):
        g = g.dropna(subset=["value"])
        if len(g) < 2:
            continue
        target = g["time"].iloc[-1] - pd.Timedelta(days=days)
        base = g.iloc[(g["time"] - target).abs().argmin()]
        rows.append({"monitoring_location_id": site,
                     f"change_{days}d_ft": g["value"].iloc[-1] - base["value"]})
    return pd.DataFrame(rows)

# restrict to wells that reported in the last 90 days
site_ids = active["monitoring_location_id"].tolist()
latest = latest_values(site_ids)
change = change_over_window(site_ids, days=30)

snapshot = (latest.merge(change, on="monitoring_location_id", how="left")
            .merge(co_sites[show], on="monitoring_location_id", how="left"))
snapshot["generated_utc"] = pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")
print(f"{len(snapshot)} wells in snapshot")
snapshot.head()

In [ ]:
out_dir = Path("data")
out_dir.mkdir(exist_ok=True)
snapshot.to_csv(out_dir / "co_gw_latest.csv", index=False)
snapshot.to_json(out_dir / "co_gw_latest.json", orient="records", date_format="iso", indent=2)
print("Wrote", out_dir / "co_gw_latest.csv", "and", out_dir / "co_gw_latest.json")